# Regularne uruchamianie skryptu .py na Windows — harmonogram, weryfikacja, centralna historia

Trzy osobne, ale powiązane tematy:

1. **Jak zaplanować** regularne uruchamianie skryptu (Harmonogram zadań Windows).
2. **Jak zweryfikować pojedyncze uruchomienie** — czy się wykonało, z jakim wynikiem.
3. **Jak mieć JEDNO miejsce** z historią wykonań wielu różnych skryptów — to
   ostatnie wymaga własnego, prostego mechanizmu, bo wbudowana historia
   Harmonogramu zadań do tego się nie nadaje (sekcja 5 wyjaśnia dlaczego).

To nie jest notebook do uruchamiania w Jupyterze na Linuksie — sekcje 1-4
to instrukcje/polecenia Windows, sekcje 6-8 to Python, który **faktycznie
przetestowałem** (logika działa niezależnie od OS, tylko harmonogram
uruchamiania jest specyficzny dla Windows).

## 1. Harmonogram zadań (Task Scheduler) — GUI krok po kroku

`Win + R` → `taskschd.msc` → **Utwórz zadanie podstawowe** (Create Basic
Task) daje kreator, ale dla skryptu, który ma działać niezawodnie,
lepiej użyć **Utwórz zadanie** (Create Task...) — daje dostęp do wszystkich
opcji od razu, bez wracania do edycji.

### Zakładka General

| Pole | Wymagane? | Komentarz |
|---|---|---|
| Nazwa | tak | np. `ETL_Klienci_Codziennie` — czytelna, bo będziesz ją widzieć w historii |
| Uruchom niezależnie od tego, czy użytkownik jest zalogowany | zalecane dla serwera | inaczej zadanie NIE odpali się, jeśli nikt nie jest zalogowany na pulpicie |
| Uruchom z najwyższymi uprawnieniami | tylko jeśli skrypt tego wymaga | (np. zapis do `C:\Program Files\...`) — domyślnie zostaw wyłączone |
| Skonfiguruj dla | zostaw domyślne | wersja Windows/Windows Server |

### Zakładka Triggers (Wyzwalacze) → New...

| Pole | Wymagane? | Komentarz |
|---|---|---|
| Begin the task | tak | `On a schedule` dla regularnego uruchamiania |
| Settings (Daily/Weekly/...) | tak | częstotliwość |
| **Advanced → Repeat task every** | opcjonalne | **jedyny sposób** na odstępy krótsze niż dzienne (np. co 15 minut) — ustaw `for a duration of: Indefinitely` |
| Enabled | tak (domyślnie zaznaczone) | łatwo przeoczyć po edycji |

### Zakładka Actions → New...

| Pole | Wymagane? | Komentarz |
|---|---|---|
| Action | tak | `Start a program` |
| Program/script | tak | **pełna ścieżka do `python.exe` z Twojego środowiska (venv!)**, nie samo `python` — patrz sekcja 3 |
| Add arguments | tak (dla skryptu) | ścieżka do pliku `.py`, w cudzysłowie jeśli zawiera spacje |
| **Start in (optional)** | **w praktyce wymagane** | katalog roboczy skryptu — najczęstsza przyczyna "działa ręcznie, nie działa z harmonogramu" (sekcja 3) |

### Zakładka Conditions

| Pole | Domyślnie | Kiedy zmienić |
|---|---|---|
| Start the task only if the computer is on AC power | zaznaczone | **odznacz na laptopie** — inaczej zadanie nie odpali się na baterii |
| Wake the computer to run this task | odznaczone | zaznacz, jeśli komputer bywa uśpiony o zaplanowanej porze |

### Zakładka Settings

| Pole | Zalecane | Komentarz |
|---|---|---|
| Allow task to be run on demand | zaznaczone | pozwala ręcznie odpalić do testu (sekcja 4) |
| If the task fails, restart every: | np. 5 min, 3 próby | automatyczny retry — przydatne przy chwilowych awariach (np. baza chwilowo niedostępna) |
| Stop the task if it runs longer than: | ustaw rozsądny limit | zabezpieczenie przed zawieszonym procesem blokującym kolejne uruchomienia |
| If the running task does not end when requested, force it to stop | zaznaczone | domyka poprzedni punkt |

## 2. To samo z linii poleceń — `schtasks`

Przydatne, gdy chcesz mieć **powtarzalne, wersjonowane w repo** tworzenie
zadania (np. skrypt wdrożeniowy dla nowego serwera) zamiast klikania w GUI
za każdym razem.

In [1]:
# UWAGA: to jest polecenie Windows (cmd/PowerShell), nie Python — pokazane jako tekst do skopiowania,
# nie do uruchomienia w tym notebooku.

polecenie_schtasks = r'''
schtasks /create ^
  /tn "ETL_Klienci_Codziennie" ^
  /tr "\"C:\Projekty\etl\venv\Scripts\python.exe\" \"C:\Projekty\etl\run_etl.py\"" ^
  /sc daily ^
  /st 05:00 ^
  /ru "DOMENA\uzytkownik_serwisowy" ^
  /rl highest
'''
print(polecenie_schtasks)


schtasks /create ^
  /tn "ETL_Klienci_Codziennie" ^
  /tr "\"C:\Projekty\etl\venv\Scripts\python.exe\" \"C:\Projekty\etl\run_etl.py\"" ^
  /sc daily ^
  /st 05:00 ^
  /ru "DOMENA\uzytkownik_serwisowy" ^
  /rl highest



Objaśnienie flag: `/tn` nazwa zadania (**wymagane**), `/tr` polecenie do
uruchomienia (**wymagane** — ścieżka do `python.exe` + ścieżka do skryptu,
oba w escapowanych cudzysłowach), `/sc` typ harmonogramu (**wymagane**:
`MINUTE`, `HOURLY`, `DAILY`, `WEEKLY`, `ONSTART`, `ONLOGON`...), `/mo`
modyfikator częstotliwości (opcjonalny, np. `/sc minute /mo 15` = co 15
minut), `/st` godzina startu (opcjonalna, format `HH:MM`), `/ru` konto,
na którym ma działać (opcjonalne — domyślnie bieżący użytkownik; na
serwerze lepiej dedykowane konto serwisowe), `/rl highest` = uruchom z
podwyższonymi uprawnieniami (opcjonalne).

`schtasks` **nie ma** odpowiednika pola "Start in" — katalog roboczy
ustawiasz albo w skrypcie `.bat`/`.ps1`, który robi `cd /d "ścieżka"` przed
wywołaniem Pythona, albo (lepiej, patrz sekcja 3) piszesz skrypt Pythona
tak, żeby w ogóle nie zależał od katalogu roboczego.

In [2]:
# przydatne polecenia pomocnicze — też do uruchomienia w cmd/PowerShell na Windows, nie tutaj

polecenia_pomocnicze = r'''
REM uruchom zadanie natychmiast, do testu (nie czekając na harmonogram)
schtasks /run /tn "ETL_Klienci_Codziennie"

REM sprawdz status i ostatni wynik (sekcja 4)
schtasks /query /tn "ETL_Klienci_Codziennie" /v /fo LIST

REM usun zadanie
schtasks /delete /tn "ETL_Klienci_Codziennie" /f
'''
print(polecenia_pomocnicze)


REM uruchom zadanie natychmiast, do testu (nie czekając na harmonogram)
schtasks /run /tn "ETL_Klienci_Codziennie"

REM sprawdz status i ostatni wynik (sekcja 4)
schtasks /query /tn "ETL_Klienci_Codziennie" /v /fo LIST

REM usun zadanie
schtasks /delete /tn "ETL_Klienci_Codziennie" /f



## 3. Najczęstsze przyczyny "działa ręcznie, nie działa z harmonogramu"

1. **Zły interpreter Pythona** — Harmonogram zadań nie zna Twojego venv.
   `Program/script` musi wskazywać na
   `C:\Projekty\etl\venv\Scripts\python.exe`, nie na samo `python`
   (które w kontekście zadania systemowego może w ogóle nie być w PATH).
2. **Zły katalog roboczy** — skrypt z relatywną ścieżką (`pd.read_csv("dane/plik.csv")`)
   działa w VS Code (bo katalog roboczy to folder projektu), ale nie z
   Harmonogramu zadań (domyślny katalog roboczy to `C:\Windows\System32`).
   Dwa rozwiązania: ustaw pole **Start in** na folder skryptu, ALBO —
   solidniej — spraw, żeby skrypt sam nie zależał od katalogu roboczego:

In [3]:
# przykładowy fragment na początek skryptu .py — nie uruchamiany tutaj (notebook nie ma __file__),
# ale to jest dokładnie kod do wklejenia na start prawdziwego skryptu
przyklad_kodu = '''
from pathlib import Path

KATALOG_SKRYPTU = Path(__file__).resolve().parent
sciezka_konfiguracji = KATALOG_SKRYPTU / "config.yaml"
sciezka_logow = KATALOG_SKRYPTU / "logi" / "historia_wykonan.log"

# działa identycznie niezależnie od tego, skąd i jak skrypt zostanie uruchomiony
'''
print(przyklad_kodu)


from pathlib import Path

KATALOG_SKRYPTU = Path(__file__).resolve().parent
sciezka_konfiguracji = KATALOG_SKRYPTU / "config.yaml"
sciezka_logow = KATALOG_SKRYPTU / "logi" / "historia_wykonan.log"

# działa identycznie niezależnie od tego, skąd i jak skrypt zostanie uruchomiony



3. **Brak uprawnień do zasobów** — konto, na którym działa zadanie (domyślnie
   `SYSTEM` albo konto, które je utworzyło), może nie mieć dostępu do
   dysku sieciowego czy bazy danych, do których masz dostęp na swoim
   koncie. Ustaw jawnie `/ru` (albo pole "Change User or Group..." w GUI)
   na konto z odpowiednimi uprawnieniami.
4. **Zmienne środowiskowe** — zadanie systemowe nie ładuje profilu
   użytkownika (np. zmiennych z `.env` ustawionych w PowerShell profile).
   Trzymaj konfigurację w pliku (`config.yaml`/`.env` czytany jawnie przez
   `python-dotenv`), nie w zmiennych środowiskowych sesji.

## 4. Weryfikacja pojedynczego uruchomienia

### a) Historia w GUI Harmonogramu zadań

Domyślnie **wyłączona**. Włącz: panel po prawej → **Enable All Tasks History**.
Dopiero wtedy zakładka **History** dla konkretnego zadania pokazuje zdarzenia
(`Task Started`, `Task Completed`, `Action failed` itd.) z dokładnym czasem.

Na liście zadań (widok główny) kluczowe kolumny:
- **Last Run Time** / **Next Run Time**
- **Last Run Result** — `0x0` = sukces; jakikolwiek inny kod = błąd (kod
  odpowiada kodowi wyjścia procesu — patrz punkt (c) niżej)

### b) PowerShell — to samo, ale skryptowalne

```powershell
Get-ScheduledTaskInfo -TaskName "ETL_Klienci_Codziennie" |
    Select-Object LastRunTime, LastTaskResult, NextRunTime
```

### c) Kod wyjścia skryptu Pythona — fundament całej weryfikacji

Nieobsłużony wyjątek w Pythonie kończy proces kodem `1` automatycznie —
Harmonogram zadań (i każdy inny scheduler, w tym Airflow z poprzedniego
notebooka) rozpoznaje to jako niepowodzenie. Nie musisz ręcznie wołać
`sys.exit()` przy normalnym wyjątku — tylko gdy chcesz **celowo** zgłosić
niepowodzenie bez wyjątku (np. "0 nowych rekordów, to podejrzane, oznacz
jako błąd") albo określony, niestandardowy kod (`sys.exit(2)` dla innego
typu awarii niż `sys.exit(1)`).

## 5. Dlaczego wbudowana historia Harmonogramu zadań TO ZA MAŁO

Dla pojedynczego skryptu na jednej maszynie — wystarcza. Dla inżyniera
danych z kilkunastoma skryptami (ETL, raporty, synchronizacje) problem:

- Historia jest **per-maszyna** — jeśli skrypty rozproszone są na kilku
  serwerach, i tak musisz sprawdzać każdy osobno.
- Nie da się **łatwo przeszukać/przefiltrować** (np. "pokaż wszystkie
  awarie w tym tygodniu") ani zrobić z tego wykresu/raportu.
- Nie ma miejsca na **szczegóły biznesowe** ("przetworzono 14 532 wiersze",
  "0 nowych rekordów — sprawdź źródło") — tylko sukces/porażka procesu.

Rozwiązanie: każdy skrypt na końcu zapisuje jeden wiersz do **wspólnej
tabeli w SQL Server** (ten sam `mssql_python` z wcześniejszego notebooka).
Jedna tabela = jedno miejsce, przeszukiwalne SQL-em, gotowe pod prosty
raport w Power BI.

## 6. Wspólna tabela historii wykonań

```sql
CREATE TABLE dbo.HistoriaWykonanSkryptow (
    id              INT IDENTITY PRIMARY KEY,
    nazwa_skryptu   NVARCHAR(200) NOT NULL,
    host            NVARCHAR(100) NOT NULL,
    start_czas      DATETIME2 NOT NULL,
    czas_trwania_s  DECIMAL(10,2) NOT NULL,
    status          NVARCHAR(20) NOT NULL,      -- 'sukces' / 'blad'
    komunikat_bledu NVARCHAR(MAX) NULL,
    liczba_rekordow INT NULL                     -- opcjonalnie: szczegół biznesowy
);
```

## 7. Reużywalny wrapper w Pythonie — `LogWykonaniaSkryptu`

To jedyny fragment w tym notebooku wart trzymania jako współdzielony kod
(np. `wspolne/logowanie.py`, importowany przez każdy skrypt) — dokładnie
ta sama logika (start, koniec, czas trwania, status, obsługa wyjątku)
wraca w KAŻDYM skrypcie, który ma się regularnie uruchamiać.

Dwuwarstwowe logowanie: **plik lokalny** (`logging`, działa zawsze, nawet
gdy SQL Server jest niedostępny) + **centralna tabela SQL** (do przeglądu
"w jednym miejscu"). Błąd zapisu do SQL nigdy nie przesłania oryginalnego
błędu skryptu — trafia tylko do loga lokalnego jako ostrzeżenie.

In [4]:
import logging
import socket
import time
import traceback
from datetime import datetime, timezone
from pathlib import Path

logging.basicConfig(
    filename=Path(__file__).resolve().parent / "historia_wykonan.log" if "__file__" in dir() else "historia_wykonan.log",
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)
logger = logging.getLogger("historia_wykonan")


class LogWykonaniaSkryptu:
    """
    Context manager do owinięcia głównej logiki skryptu:

        with LogWykonaniaSkryptu("etl_klienci", connection_string=CONN_STR):
            ... główna logika skryptu ...

    WYMAGANE: nazwa_skryptu (do identyfikacji w tabeli/logu).
    OPCJONALNE: connection_string — jeśli brak, loguje tylko lokalnie do pliku.
    Wyjątek z bloku 'with' jest logowany, a potem PUSZCZANY DALEJ (return False
    w __exit__) — dzięki temu proces kończy się kodem 1, co Harmonogram
    zadań poprawnie rozpozna jako niepowodzenie (sekcja 4c).
    """

    def __init__(self, nazwa_skryptu: str, connection_string: str | None = None):
        self.nazwa_skryptu = nazwa_skryptu
        self.connection_string = connection_string
        self.host = socket.gethostname()

    def __enter__(self):
        self._start_monotonic = time.monotonic()
        self._start_ts = datetime.now(timezone.utc)
        logger.info(f"START {self.nazwa_skryptu} (host={self.host})")
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        czas_trwania = round(time.monotonic() - self._start_monotonic, 2)

        if exc_type is None:
            status, komunikat = "sukces", None
            logger.info(f"KONIEC {self.nazwa_skryptu} - sukces ({czas_trwania}s)")
        else:
            status = "blad"
            komunikat = f"{exc_type.__name__}: {exc_val}"
            logger.error(f"KONIEC {self.nazwa_skryptu} - BLAD ({czas_trwania}s): {komunikat}")
            logger.error(traceback.format_exc())

        self._zapisz_do_sql(status, czas_trwania, komunikat)
        return False   # NIE tłumimy wyjątku — ma polecieć dalej

    def _zapisz_do_sql(self, status: str, czas_trwania: float, komunikat: str | None) -> None:
        if not self.connection_string:
            return
        try:
            from mssql_python import connect
            with connect(self.connection_string) as conn:
                cursor = conn.cursor()
                cursor.execute(
                    """
                    INSERT INTO dbo.HistoriaWykonanSkryptow
                        (nazwa_skryptu, host, start_czas, czas_trwania_s, status, komunikat_bledu)
                    VALUES (%(nazwa)s, %(host)s, %(start)s, %(czas)s, %(status)s, %(komunikat)s)
                    """,
                    {
                        "nazwa": self.nazwa_skryptu, "host": self.host, "start": self._start_ts,
                        "czas": czas_trwania, "status": status, "komunikat": komunikat,
                    },
                )
                conn.commit()
        except Exception as blad_zapisu:
            # zapis do SQL nie może przesłonić oryginalnego błędu skryptu — tylko ostrzeżenie lokalnie
            logger.warning(f"Nie udalo sie zapisac historii do SQL: {blad_zapisu}")

## 8. Test wrappera — przypadek sukcesu i przypadek błędu

Testujemy warstwę, która działa niezależnie od Windows i SQL Server:
logowanie lokalne + poprawne puszczanie wyjątku dalej (czyli: kod wyjścia
procesu). Zapis do SQL pomijamy (`connection_string=None`) — ta część
korzysta z dokładnie tego samego, już zweryfikowanego API `mssql_python`
co poprzedni notebook.

In [5]:
# przypadek sukcesu
with LogWykonaniaSkryptu("test_sukces"):
    suma = sum(range(1000))
    print(f"Przetworzono, wynik: {suma}")

Przetworzono, wynik: 499500


In [6]:
# przypadek błędu — wyjątek MUSI polecieć dalej (sprawdzamy to jawnie)
try:
    with LogWykonaniaSkryptu("test_blad"):
        raise ValueError("Symulowany błąd: brak połączenia ze źródłem danych")
except ValueError as e:
    print(f"Wyjątek poprawnie doleciał do zewnętrznego kodu: {e}")

Wyjątek poprawnie doleciał do zewnętrznego kodu: Symulowany błąd: brak połączenia ze źródłem danych


In [7]:
# zawartość loga lokalnego po obu przebiegach — dokładnie to, co trafiłoby do pliku na serwerze
print(Path("historia_wykonan.log").read_text(encoding="utf-8"))

2026-09-12 10:11:31,519 | INFO | START test_sukces (host=vm)
2026-09-12 10:11:31,519 | INFO | KONIEC test_sukces - sukces (0.0s)
2026-09-12 10:11:31,524 | INFO | START test_blad (host=vm)
2026-09-12 10:11:31,525 | ERROR | KONIEC test_blad - BLAD (0.0s): ValueError: Symulowany błąd: brak połączenia ze źródłem danych
2026-09-12 10:11:31,525 | ERROR | Traceback (most recent call last):
  File "/tmp/ipykernel_113/217498730.py", line 4, in <module>
    raise ValueError("Symulowany błąd: brak połączenia ze źródłem danych")
ValueError: Symulowany błąd: brak połączenia ze źródłem danych




Ostatni test, kluczowy dla Harmonogramu zadań: gdy skrypt z `with LogWykonaniaSkryptu(...)`
zawiera nieobsłużony wyjątek i jest uruchomiony jako osobny proces (nie
`try/except` jak wyżej, tylko cały skrypt `.py` uruchomiony przez
`python.exe`), proces musi zakończyć się kodem **innym niż 0**.

In [8]:
import subprocess
import sys

# demo na samodzielnym, uproszczonym skrypcie (żeby test nie zależał od importu z tego notebooka)
kod_skryptu = '''
class LogWykonaniaSkryptu:
    def __enter__(self):
        return self
    def __exit__(self, exc_type, exc_val, exc_tb):
        return False

with LogWykonaniaSkryptu():
    raise ValueError("symulowana awaria")
'''

with open("demo_skrypt_z_bledem.py", "w", encoding="utf-8") as f:
    f.write(kod_skryptu)

wynik = subprocess.run([sys.executable, "demo_skrypt_z_bledem.py"], capture_output=True, text=True)
print(f"Kod wyjścia procesu: {wynik.returncode}")   # oczekiwane: 1 (nieobsłużony wyjątek)
print("stderr (fragment):", wynik.stderr.strip().splitlines()[-1])

Kod wyjścia procesu: 1
stderr (fragment): ValueError: symulowana awaria


Kod wyjścia `1` to dokładnie to, co Harmonogram zadań pokaże jako
`Last Run Result` różny od `0x0` — cały łańcuch weryfikacji (sekcja 4)
zamyka się spójnie: wyjątek w skrypcie → niezerowy kod wyjścia procesu →
Harmonogram zadań oznacza uruchomienie jako nieudane → jednocześnie wiersz
w `dbo.HistoriaWykonanSkryptow` ma `status='blad'` ze szczegółowym
komunikatem, którego Harmonogram zadań by nie pokazał.

## 9. Zapytanie SQL — "jedno miejsce" z historią wszystkich skryptów

Przykładowe zapytania, które dają dokładnie to, o co chodziło w pytaniu —
przegląd stanu wszystkich regularnych skryptów naraz.

In [9]:
zapytanie_ostatnie_uruchomienie = '''
-- ostatnie uruchomienie KAŻDEGO skryptu, z jego statusem
SELECT
    nazwa_skryptu,
    MAX(start_czas) AS ostatnie_uruchomienie,
    (SELECT TOP 1 status FROM dbo.HistoriaWykonanSkryptow h2
     WHERE h2.nazwa_skryptu = h1.nazwa_skryptu
     ORDER BY start_czas DESC) AS ostatni_status
FROM dbo.HistoriaWykonanSkryptow h1
GROUP BY nazwa_skryptu
ORDER BY ostatnie_uruchomienie DESC;
'''

zapytanie_awarie_tydzien = '''
-- wszystkie awarie w ostatnich 7 dniach, z komunikatem błędu
SELECT nazwa_skryptu, start_czas, czas_trwania_s, komunikat_bledu
FROM dbo.HistoriaWykonanSkryptow
WHERE status = 'blad'
  AND start_czas >= DATEADD(day, -7, GETUTCDATE())
ORDER BY start_czas DESC;
'''

zapytanie_trend_czasu = '''
-- czy skrypt zaczyna trwać coraz dłużej (sygnał rosnącego wolumenu danych / degradacji wydajności)
SELECT
    nazwa_skryptu,
    CAST(start_czas AS DATE) AS dzien,
    AVG(czas_trwania_s) AS sredni_czas_s
FROM dbo.HistoriaWykonanSkryptow
WHERE nazwa_skryptu = 'etl_klienci'
GROUP BY nazwa_skryptu, CAST(start_czas AS DATE)
ORDER BY dzien;
'''

print("Trzy gotowe zapytania — przydatne wprost jako źródło dla tabeli/karty w Power BI.")

Trzy gotowe zapytania — przydatne wprost jako źródło dla tabeli/karty w Power BI.


## 10. Powiadomienia o błędach (opcjonalnie, ale warto)

Najprostszy wariant — bez dodatkowej infrastruktury — e-mail wysyłany z
poziomu `except` w samym skrypcie (np. `smtplib` + serwer SMTP firmowy),
albo webhook na Teams/Slacka. Jeśli już korzystasz z Airflow do części
pipeline'ów (poprzedni notebook) — tam `on_failure_callback` załatwia to
samo bez pisania tego ręcznie w każdym skrypcie. Dla skryptów spoza
Airflow, uruchamianych z Harmonogramu zadań: dopisz wysyłkę powiadomienia
w bloku `_zapisz_do_sql`/osobnej metodzie `LogWykonaniaSkryptu`, warunkowo
gdy `status == "blad"`.

## Podsumowanie

1. **Harmonogram** — `Create Task` (nie `Create Basic Task`) w GUI albo
   `schtasks /create` w CLI. Najczęstsza pułapka: zły interpreter (pełna
   ścieżka do `venv\Scripts\python.exe`) i zły katalog roboczy (ustaw
   `Start in`, ALBO — solidniej — użyj `Path(__file__).parent` w skrypcie).
2. **Weryfikacja pojedynczego uruchomienia** — włącz `Enable All Tasks
   History`, sprawdzaj `Last Run Result` (`0x0` = sukces) albo
   `Get-ScheduledTaskInfo` z PowerShell. Kod wyjścia procesu Pythona
   (automatyczny `1` przy nieobsłużonym wyjątku) to fundament, na którym
   to wszystko się opiera.
3. **Jedno miejsce z historią wielu skryptów** — wbudowana historia
   Harmonogramu do tego nie wystarcza (per-maszyna, brak szczegółów
   biznesowych, nieprzeszukiwalna). Własna tabela SQL Server + reużywalny
   context manager (`LogWykonaniaSkryptu`) w każdym skrypcie daje dokładnie
   to — i od razu gotowe źródło pod raport w Power BI.